# 循环与分支结构实现ReAct

1. 静态通过添加条件边add_conditional_edges，提供路由，path-math来实现
2. 通过Command通过goto来实现

![](./resources/langgraph_ReAct.svg)

In [18]:
import os
from datetime import datetime
from pprint import pprint
from random import randint
from typing import Literal

from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolCall, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.constants import END, START
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState
from loguru import logger
from utils.util import show_md

load_dotenv(override=True)

True

In [19]:
@tool(parse_docstring=True)
def get_weather(city: str = "深圳"):
    """
    查询指定城市当日天气

    Args:
        city: 要查询天气的城市名称，不传则默认使用深圳。
    """
    return f"{city}天气晴朗，气温32℃，体感温度达37℃，伴有2级北风。"

In [20]:
pprint(get_weather.args_schema.model_json_schema())

{'description': '查询指定城市当日天气',
 'properties': {'city': {'default': '深圳',
                         'description': '要查询天气的城市名称，不传则默认使用深圳。',
                         'title': 'City',
                         'type': 'string'}},
 'title': 'get_weather',
 'type': 'object'}


`parse_docstring=True`，就相当于给 LangChain 下达了一个指令：“请用 Google Style 规则来解析我的注释，不要瞎猜”。所以：

- **必须写 `Args:`**。
- **参数名 + 冒号 `:`** 是强制格式。
- 这样解析出的 JSON Schema 才是最干净、最准确的。当以后你把一堆工具扔给大模型（如 `llm.bind_tools([get_weather])`）时，大模型看到的就是你截图里这份极其规范的 JSON 结构，它调用工具的准确率就会大幅提升。


In [21]:
@tool(parse_docstring=True)
def get_news(domain: Literal["AI", "食品安全"]):
    """
    查询特定领域的当日热点

    Args:
        domain: 特定领域
    """
    if domain == "AI":
        return "DeepSeek宣布全面开源内部推理框架DeepInfer，并同步开源生产级Token编解码组件deepseek-recipe。"
    else:
        return "过去依赖进口的牛油果、藜麦、鹰嘴豆等“洋食材”正通过农业科技、产业机制和政策支持实现本土化种植，成为助力中国乡村增收致富的土特产。"

In [22]:
pprint(get_news.args_schema.model_json_schema())

{'description': '查询特定领域的当日热点',
 'properties': {'domain': {'description': '特定领域',
                           'enum': ['AI', '食品安全'],
                           'title': 'Domain',
                           'type': 'string'}},
 'required': ['domain'],
 'title': 'get_news',
 'type': 'object'}


In [23]:
@tool(parse_docstring=True)
def get_current_time(timezone: str = "Asia/Shanghai"):
    """
    获取指定时区的当前时间

    Args:
        timezone: 要查询的时区名称（如 Asia/Shanghai, UTC, America/New_York），不传则默认使用北京时间。
    """
    # 这里为了演示，直接返回一个模拟的格式化时间字符串
    # 如果需要真实解析时区，可以使用 zoneinfo 或 pytz 库
    now = datetime.now()
    return f"当前 {timezone} 时间是 {now.strftime('%Y-%m-%d %H:%M:%S')}，请注意合理安排时间。"

In [24]:
tools_map = {tool.name: tool for tool in [get_news, get_weather, get_current_time]}


def handle_tool_call(tool_call: ToolCall):
    tool_name = tool_call["name"]
    tool = tools_map.get(tool_name)

    fail_prob = 6

    if not tool_call:
        logger.warning(f"{tool_name}未实现")
        return ToolMessage(
            content=f"Error: 未实现{tool_name}工具", tool_call_id=tool_call["id"]
        )

    elif randint(0, 9) < fail_prob:
        logger.error(f"由于网络波动，调用{tool_name}工具失败")
        return ToolMessage(
            content=f"网络波动，调用{tool_name}工具失败，请重试",
            tool_call_id=tool_call["id"],
        )
    else:
        tool_msg = tool.invoke(tool_call)
        logger.info(f"调用工具{tool_name}成功: {type[tool_msg]} {tool_msg}")
        return tool_msg


In [25]:
llm = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

llm_with_tools = llm.bind_tools([get_weather, get_news, get_current_time])

# 路由的实现

In [26]:
class ReActState(MessagesState):
    user_input: str
    output: str


def input_node(state: ReActState, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 input_node")
    return {"messages": [HumanMessage(state["user_input"])]}


def llm_node(state: ReActState, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 llm_node")
    ai_message = llm_with_tools.invoke(state["messages"])
    return {"messages": [ai_message]}


def tool_node(state: ReActState, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 tool_node")
    messages = state["messages"]

    ai_msg = messages[-1]
    messages = []

    for tool_call in ai_msg.tool_calls:
        tool_msg = handle_tool_call(tool_call)
        messages.append(tool_msg)

    return {"messages": messages}


def answer_node(state: ReActState, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 answer_node")
    msg = state["messages"][-1]
    return {"output": msg.content}


def router(
    state: ReActState, config: RunnableConfig
) -> Literal["tool_node", "answer_node"]:
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 router")

    msg = state["messages"][-1]
    return "tool_node" if msg.tool_calls else "answer_node"


In [27]:
graph_builder = StateGraph(state_schema=ReActState)
graph_builder.add_node(input_node)
graph_builder.add_node(llm_node)
graph_builder.add_node(tool_node)
graph_builder.add_node(answer_node)

graph_builder.add_edge(START, "input_node")
graph_builder.add_edge("input_node", "llm_node")
graph_builder.add_conditional_edges("llm_node", router)
graph_builder.add_edge("tool_node", "llm_node")
graph_builder.add_edge("answer_node", END)

graph = graph_builder.compile()

In [28]:
res = graph.invoke(
    {
        "user_input": "现在是几号？帮我查询深圳现在的天气和今天的AI热点",
        "messages": [
            SystemMessage("如果工具调用失败，必须重新调用直至成功"),
            SystemMessage(
                "添加合适的emoji风格并且请直接回答用户的最终问题，绝对不要向用户提及工具调用失败、网络波动或重试等内部执行细节。"
            ),
        ],
    }
)

2026-10-21 13:46:34.619 | INFO     | __main__:input_node:8 - 当前超步1 执行 input_node
2026-10-21 13:46:34.621 | INFO     | __main__:llm_node:14 - 当前超步2 执行 llm_node
2026-10-21 13:46:36.613 | INFO     | __main__:router:45 - 当前超步2 执行 router
2026-10-21 13:46:36.614 | INFO     | __main__:tool_node:21 - 当前超步3 执行 tool_node
2026-10-21 13:46:36.615 | INFO     | __main__:handle_tool_call:24 - 调用工具get_current_time成功: type[ToolMessage(content='当前 Asia/Shanghai 时间是 2026-10-21 13:46:36，请注意合理安排时间。', name='get_current_time', tool_call_id='01a15cff6cb998637136a6101936b334')] content='当前 Asia/Shanghai 时间是 2026-10-21 13:46:36，请注意合理安排时间。' name='get_current_time' tool_call_id='01a15cff6cb998637136a6101936b334'
2026-10-21 13:46:36.616 | INFO     | __main__:handle_tool_call:24 - 调用工具get_weather成功: type[ToolMessage(content='深圳天气晴朗，气温32℃，体感温度达37℃，伴有2级北风。', name='get_weather', tool_call_id='01a15cff6cb998637136a6101936b335')] content='深圳天气晴朗，气温32℃，体感温度达37℃，伴有2级北风。' name='get_weather' tool_call_id='01a15cff6cb9986371

In [29]:
show_md(res["output"])

📅 **今天是 2026年10月21日（星期三）**，现在北京时间 13:46 ⏰

🌤️ **深圳天气**
- 天气：晴朗 ☀️
- 气温：32℃ 🌡️
- 体感温度：37℃ 🔥（有点闷热，注意防晒补水）
- 风：2级北风 🍃

🤖 **今日 AI 热点**
- DeepSeek 宣布全面开源内部推理框架 **DeepInfer**，并同步开源生产级 Token 编解码组件 **deepseek-recipe** 🚀

出门记得带伞防晒，午后高温时段尽量减少户外活动哦～ 😎

In [30]:
_ = [msg.pretty_print() for msg in res["messages"]]

================================ System Message ================================

如果工具调用失败，必须重新调用直至成功
================================ System Message ================================

添加合适的emoji风格并且请直接回答用户的最终问题，绝对不要向用户提及工具调用失败、网络波动或重试等内部执行细节。
================================ Human Message =================================

现在是几号？帮我查询深圳现在的天气和今天的AI热点
================================== Ai Message ==================================
Tool Calls:
  get_current_time (01a15cff6cb998637136a6101936b334)
 Call ID: 01a15cff6cb998637136a6101936b334
  Args:
    timezone: Asia/Shanghai
  get_weather (01a15cff6cb998637136a6101936b335)
 Call ID: 01a15cff6cb998637136a6101936b335
  Args:
    city: 深圳
  get_news (01a15cff6cb998637136a6101936b336)
 Call ID: 01a15cff6cb998637136a6101936b336
  Args:
    domain: AI
================================= Tool Message =================================
Name: get_current_time

当前 Asia/Shanghai 时间是 2026-10-21 13:46:36，请注意合理安排时间。
================================= Tool Me

# 动态Command goto实现


上面的是通过`graph_builder.add_conditional_edges("llm_node", router)`的方式实现的

```python
def router(
    state: ReActState, config: RunnableConfig
) -> Literal["tool_node", "answer_node"]:
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 router")

    msg = state["messages"][-1]
    return "tool_node" if msg.tool_calls else "answer_node"
```

使用Command来实现

In [31]:
from langgraph.types import Command


def llm_node(
    state: ReActState, config: RunnableConfig
) -> Command[Literal["tool_node", "answer_node"]]:
    cur_step = config["metadata"]["langgraph_step"]
    logger.info(f"当前超步{cur_step} 执行 llm_node")
    ai_message = llm_with_tools.invoke(state["messages"])

    goto = "tool_node" if ai_message.tool_calls else "answer_node"

    return Command(update={"messages": [ai_message]}, goto=goto)


In [32]:
graph_builder = StateGraph(state_schema=ReActState)
graph_builder.add_node(input_node)
graph_builder.add_node(llm_node)
graph_builder.add_node(tool_node)
graph_builder.add_node(answer_node)

graph_builder.add_edge(START, "input_node")
graph_builder.add_edge("input_node", "llm_node")
graph_builder.add_edge("tool_node", "llm_node")
graph_builder.add_edge("answer_node", END)

graph = graph_builder.compile()

In [33]:
res = graph.invoke(
    {
        "user_input": "现在是几号？帮我查询深圳现在的天气和今天的AI热点",
        "messages": [
            SystemMessage("如果工具调用失败，必须重新调用直至成功"),
            SystemMessage(
                "添加合适的emoji风格并且请直接回答用户的最终问题，绝对不要向用户提及工具调用失败、网络波动或重试等内部执行细节。"
            ),
        ],
    }
)

2026-10-21 13:46:40.167 | INFO     | __main__:input_node:8 - 当前超步1 执行 input_node
2026-10-21 13:46:40.169 | INFO     | __main__:llm_node:8 - 当前超步2 执行 llm_node
2026-10-21 13:46:42.060 | INFO     | __main__:tool_node:21 - 当前超步3 执行 tool_node
2026-10-21 13:46:42.060 | ERROR    | __main__:handle_tool_call:17 - 由于网络波动，调用get_current_time工具失败
2026-10-21 13:46:42.061 | ERROR    | __main__:handle_tool_call:17 - 由于网络波动，调用get_weather工具失败
2026-10-21 13:46:42.061 | ERROR    | __main__:handle_tool_call:17 - 由于网络波动，调用get_news工具失败
2026-10-21 13:46:42.062 | INFO     | __main__:llm_node:8 - 当前超步4 执行 llm_node
2026-10-21 13:46:44.268 | INFO     | __main__:tool_node:21 - 当前超步5 执行 tool_node
2026-10-21 13:46:44.269 | ERROR    | __main__:handle_tool_call:17 - 由于网络波动，调用get_current_time工具失败
2026-10-21 13:46:44.270 | INFO     | __main__:handle_tool_call:24 - 调用工具get_weather成功: type[ToolMessage(content='深圳天气晴朗，气温32℃，体感温度达37℃，伴有2级北风。', name='get_weather', tool_call_id='01a15cff97282d5a3e4d1693772f3f40')] content='深圳

In [34]:
show_md(res["output"])

📅 **今天是 2026 年 10 月 21 日**（北京时间 13:46）

☀️ **深圳天气**
- 天气：晴朗
- 气温：32℃
- 体感温度：37℃（有点闷热哦 🥵）
- 风力：2 级北风

出门记得防晒、多补水～ 🧴💧

🤖 **今日 AI 热点**
- DeepSeek 宣布全面开源其内部推理框架 **DeepInfer**，并同步开源生产级 Token 编解码组件 **deepseek-recipe** 🚀

需要我进一步展开这条新闻，或者帮你看看别的领域（比如食品安全）的热点吗？😊

# 使用LangGraph提供的ToolNode

In [35]:
from langgraph.prebuilt.tool_node import ToolNode

graph_builder = StateGraph(state_schema=ReActState)
graph_builder.add_node(input_node)
graph_builder.add_node(llm_node)

# 不用像上面那样手动构建ToolNode
graph_builder.add_node(
    "tool_node", ToolNode(tools=[get_weather, get_news, get_current_time])
)
graph_builder.add_node(answer_node)

graph_builder.add_edge(START, "input_node")
graph_builder.add_edge("input_node", "llm_node")
graph_builder.add_edge("tool_node", "llm_node")
graph_builder.add_edge("answer_node", END)

graph = graph_builder.compile()

In [36]:
res = graph.invoke(
    {
        "user_input": "现在是几号？帮我查询深圳现在的天气和今天的AI热点",
        "messages": [
            SystemMessage("如果工具调用失败，必须重新调用直至成功"),
            SystemMessage(
                "添加合适的emoji风格并且请直接回答用户的最终问题，绝对不要向用户提及工具调用失败、网络波动或重试等内部执行细节。"
            ),
        ],
    }
)

2026-10-21 14:10:01.772 | INFO     | __main__:input_node:8 - 当前超步1 执行 input_node
2026-10-21 14:10:01.775 | INFO     | __main__:llm_node:8 - 当前超步2 执行 llm_node
2026-10-21 14:10:03.845 | INFO     | __main__:llm_node:8 - 当前超步4 执行 llm_node
2026-10-21 14:10:06.416 | INFO     | __main__:answer_node:36 - 当前超步5 执行 answer_node


In [ ]:
show_md(res["output"])

📅 **今天是 2026 年 10 月 21 日**（北京时间 14:10）

☀️ **深圳今日天气**
- 天气：晴朗
- 气温：32℃，体感高达 37℃ 🥵
- 风力：2 级北风

出门记得防晒、多补水哦～🧴💧

🤖 **今日 AI 热点**
- DeepSeek 宣布**全面开源内部推理框架 DeepInfer**，并同步开源生产级 Token 编解码组件 deepseek-recipe 🚀

简单说：又一大波开源干货来了，做推理部署和 Token 处理的小伙伴可以关注一下 👀